# **Longitudinal Administrative Healthcare Dataset for Next-Year High-Cost Member Prediction**

**The model can only use information that would already be available before the next year begins. It cannot use future information.**
```
- Utilization patterns
    Number of hospital visits
    Emergency room visits
    Doctor appointments
    Prescription usage
- Clinical signals
    Diagnosed diseases
    Chronic conditions
    Medical procedures
    Laboratory indicators (if available)
- Demographic attributes
    Age
    Gender
    Geographic region
- Network features
    Healthcare provider network
    Insurance plan
    Hospital network information
- Engineered variables
(Features created from historical data, such as:0)
    Average annual healthcare cost
    Total visits in the past year
    Cost growth rate
    Number of chronic conditions
    Average monthly claims
```

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import VotingClassifier
from deslib.des.knora_e import KNORAE # (Stacking)
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.preprocessing import LabelEncoder


import xgboost as xgb
import lightgbm as lgb
import catboost as cgb

from sklearn.metrics import accuracy_score, mean_squared_error, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import validation_curve

import warnings

warnings.filterwarnings("ignore")


# **Binary Classification**

## **PATHS & LOAD**


In [ ]:
"""
Healthcare Cost Prediction — Preprocessing Pipeline  v2
========================================================
Key changes from v1
  - NO temporal train/val split (2024 is test-only, not in training CSVs)
  - Stratified K-Fold applied AFTER target definition so folds are class-balanced
  - Returns X, y (regression), y_cls (binary HighCost), skf_splits for downstream use
  - process_main / dob / cpt / drg / icd helpers unchanged
"""

import os, warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing   import LabelEncoder

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────
BASE_DIR      = r"/home/noneo/Codes/ML/MachineLearning/Hand_On_ML/Data/Healthcare Cost Dataset"
TRAINING_PATH = os.path.join(BASE_DIR, "train", "train") + os.sep
TESTING_PATH  = os.path.join(BASE_DIR, "test",  "test")  + os.sep

COST_THRESHOLD = 30_000      # HighCost binary label
N_FOLDS        = 5
RANDOM_STATE   = 42


# ═══════════════════════════════════════════════════════════════════════════════
#  HELPER — main_df cleaning
# ═══════════════════════════════════════════════════════════════════════════════
def process_main(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Keep only yearly roll-up rows
    df = df[df["MONTH"] == -1].reset_index(drop=True)

    # Drop leakage / constant / identifier columns
    LEAKAGE_COLS = [
        "MONTH", "ActualYear", "ActualMonthNumber", "IsYTD", "QUARTER",
        "PredictedCost", "Leakage_Cost", "IsLatest",
    ]
    df.drop(columns=[c for c in LEAKAGE_COLS if c in df.columns], inplace=True)

    # Date engineering
    ref_date = pd.Timestamp("2023-01-01")
    for raw_col, new_col in [("AWVLastDate", "DaysSinceAWV"),
                              ("LastPCPVisit", "DaysSinceLastPCP")]:
        if raw_col in df.columns:
            df[raw_col] = pd.to_datetime(df[raw_col], errors="coerce")
            df[new_col] = (ref_date - df[raw_col]).dt.days.clip(lower=0)
            df.drop(columns=[raw_col], inplace=True)

    # Column groups
    cost_cols = [c for c in df.columns if c.endswith("_Cost") and c != "TotalCost"]
    count_cols = [c for c in [
        "ProviderVisitCount","ERAdmisison","30dayHospitalReadmission","OutpatientVisits",
        "Inpatient","HomeHealth","Hospice","SkilledNursingFacilities",
        "EmergencyDepartmentVisits","EmergencyDepartmentVisitsWithAdmissions",
        "CTEvents","MRIEvents","RadiologyEvents","OtherImagingServices",
        "LabEventsPathalogy","LabEventsClinicalDiagnostics","PrimaryCareServicesTotal",
        "PrimaryCareServiceswithPrimaryCarePhysician","PrimaryCareServicesWithSpecialistPhysician",
        "PrimaryCareServicesWithNursePractitioner","30DayReadmission","NewPatients",
        "EstablishedPatients","PostDischargeVisits","AmbulanceEvents","Medication",
        "PCPVisits","OfficeVisits","EDVisitsWithNoFollowUp","IPVisitsWithNoFollowUp",
        "TotalClaims","TotalScriptsFilled","AWVVisits","PediatricsVisits","UrgentCareVisit",
        "AmbulatorySurgeryVisit","DentistEvents",
    ] if c in df.columns]
    los_cols    = [c for c in df.columns if "LOS" in c]
    binary_cols = [c for c in [
        "IsCovid","DoneBySelf","AWV","Admission","InNetworkPCP","Ishighrisk",
        "AWV_Compliant","AWV_Eligible","IPPE_Eligible","IPPE_Compliant",
        "AWV_Compliant_Eligible","InNetworkAWVCompliant","OutNetworkAWVCompliant",
        "OutNetworkPCP","NonClaimBasedPayment",
    ] if c in df.columns]
    disease_cols = [c for c in [
        "ChronicObstructivePulmonaryDiseaseOrAsthma","CongestiveHeartFailure",
        "BacterialPneumonia","DiabetesShortTermComplications","DiabetesLongTermComplications",
        "UncontrolledDiabetes","AmputationDiabetes","Dialysis","ChronicConditions",
    ] if c in df.columns]
    hcc_cols = [c for c in [
        "MemberHccScore","MemberHccScoreLastYear","MemberHccScoreLastTwoYear","PersiviaMemberHccScore",
    ] if c in df.columns]
    cat_cols = [c for c in [
        "AWVCode","AWVStatus","AttributionStatus","LastPCPProvider",
        "AWVProviderNetwork","Payers_key","Enrollment_key",
    ] if c in df.columns]

    # Fill strategies
    df[cost_cols]    = df[cost_cols].fillna(0).clip(lower=0)
    df[count_cols]   = df[count_cols].fillna(0).clip(lower=0)
    df[los_cols]     = df[los_cols].fillna(0).clip(lower=0)
    df[binary_cols]  = df[binary_cols].fillna(0).replace({True:1,False:0}).astype(int)
    df[disease_cols] = df[disease_cols].fillna(0).astype(int)
    df[hcc_cols]     = df[hcc_cols].apply(lambda c: c.fillna(c.median()))

    # Frequency-encode categoricals
    for col in cat_cols:
        df[col] = df[col].astype(str).fillna("Unknown")
        freq     = df[col].value_counts(normalize=True)
        df[col]  = df[col].map(freq)

    # Fill any remaining numeric NaN
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].fillna(0)

    print(f"    main_df → {df.shape[0]:,} rows × {df.shape[1]} cols")
    return df


# ═══════════════════════════════════════════════════════════════════════════════
#  HELPER — dob_df
# ═══════════════════════════════════════════════════════════════════════════════
def process_dob(dob: pd.DataFrame, reference_year: int = 2023) -> pd.DataFrame:
    dob = dob.copy()
    dob["DOB_Key"]     = pd.to_datetime(dob["DOB_Key"], errors="coerce")
    dob["Age"]         = reference_year - dob["DOB_Key"].dt.year
    dob["Age"]         = dob["Age"].clip(lower=0).fillna(dob["Age"].median())
    dob["Gender_Key"]  = dob["Gender_Key"].fillna(dob["Gender_Key"].mode()[0]).astype(int)
    dob["AgeGroup_Key"]= dob["AgeGroup_Key"].fillna(dob["AgeGroup_Key"].mode()[0]).astype(int)
    dob.drop(columns=["DOB_Key"], inplace=True)
    print(f"    dob_df  → {dob.shape[0]:,} rows × {dob.shape[1]} cols")
    return dob


# ═══════════════════════════════════════════════════════════════════════════════
#  HELPER — cpt_df aggregation
# ═══════════════════════════════════════════════════════════════════════════════
def aggregate_cpt(cpt: pd.DataFrame) -> pd.DataFrame:
    cpt = cpt.copy()
    cpt["Procedure_Code"] = cpt["Procedure_Code"].astype(str)
    grp = cpt.groupby(["Member_Key", "StartDate"])
    agg = grp["Procedure_Code"].agg(
        cpt_ProcedureCount   ="count",
        cpt_UniqueProcedures ="nunique",
        cpt_TopProcedure     =lambda x: x.mode().iloc[0] if len(x) else "Unknown",
    ).reset_index()
    agg["cpt_ProcedureDiversity"] = agg["cpt_UniqueProcedures"] / agg["cpt_ProcedureCount"].replace(0,1)
    freq = agg["cpt_TopProcedure"].value_counts(normalize=True)
    agg["cpt_TopProcedure"] = agg["cpt_TopProcedure"].map(freq).fillna(0)
    agg.rename(columns={"StartDate":"YEAR"}, inplace=True)
    print(f"    cpt_df  → {agg.shape[0]:,} rows after aggregation")
    return agg


# ═══════════════════════════════════════════════════════════════════════════════
#  HELPER — drg_df aggregation
# ═══════════════════════════════════════════════════════════════════════════════
def aggregate_drg(drg: pd.DataFrame) -> pd.DataFrame:
    drg = drg.copy()
    drg["Code"] = drg["Code"].astype(str)
    grp = drg.groupby(["Member_Key","Start_Date_Year"])
    agg = grp["Code"].agg(
        drg_ClaimsCount="count",
        drg_UniqueDRGs ="nunique",
        drg_MostCommon =lambda x: x.mode().iloc[0] if len(x) else "Unknown",
    ).reset_index()
    drg_freq  = drg["Code"].value_counts()
    rare_drgs = set(drg_freq[drg_freq == 1].index)
    drg["is_rare"] = drg["Code"].isin(rare_drgs).astype(int)
    rare_agg  = drg.groupby(["Member_Key","Start_Date_Year"])["is_rare"].sum().reset_index()
    rare_agg.rename(columns={"is_rare":"drg_RareDRGCount"}, inplace=True)
    agg = agg.merge(rare_agg, on=["Member_Key","Start_Date_Year"], how="left")
    freq = agg["drg_MostCommon"].value_counts(normalize=True)
    agg["drg_MostCommon"] = agg["drg_MostCommon"].map(freq).fillna(0)
    agg.rename(columns={"Start_Date_Year":"YEAR"}, inplace=True)
    print(f"    drg_df  → {agg.shape[0]:,} rows after aggregation")
    return agg


# ═══════════════════════════════════════════════════════════════════════════════
#  HELPER — icd_df aggregation
# ═══════════════════════════════════════════════════════════════════════════════
def aggregate_icd(icd: pd.DataFrame) -> pd.DataFrame:
    icd = icd.copy()
    icd["Diagnosis_Code"] = icd["Diagnosis_Code"].astype(str)
    CHRONIC_PREFIXES = ("E11","E10","I50","I25","J44","J45","N18","G20","F32","F41","M79")
    icd["is_chronic"] = icd["Diagnosis_Code"].str.startswith(CHRONIC_PREFIXES).astype(int)
    grp = icd.groupby(["Member_Key","Start_Date"])
    agg = grp.agg(
        icd_DiagnosisCount       =("Diagnosis_Code","count"),
        icd_UniqueDiagnoses      =("Diagnosis_Code","nunique"),
        icd_ChronicDiagnosisCount=("is_chronic","sum"),
        icd_TopICD               =("Diagnosis_Code", lambda x: x.mode().iloc[0] if len(x) else "Unknown"),
    ).reset_index()
    agg["icd_ICDDiversity"] = agg["icd_UniqueDiagnoses"] / agg["icd_DiagnosisCount"].replace(0,1)
    freq = agg["icd_TopICD"].value_counts(normalize=True)
    agg["icd_TopICD"] = agg["icd_TopICD"].map(freq).fillna(0)
    agg.rename(columns={"Start_Date":"YEAR"}, inplace=True)
    print(f"    icd_df  → {agg.shape[0]:,} rows after aggregation")
    return agg


# ═══════════════════════════════════════════════════════════════════════════════
#  HELPER — merge all tables
# ═══════════════════════════════════════════════════════════════════════════════
def build_feature_table(main, dob, cpt, drg, icd) -> pd.DataFrame:
    for df in [main, cpt, drg, icd]:
        df["Member_Key"] = df["Member_Key"].astype(int)
        if "YEAR" in df.columns:
            df["YEAR"] = df["YEAR"].astype(int)
    dob["Member_Key"] = dob["Member_Key"].astype(int)

    merged = main.merge(dob, on="Member_Key",          how="left")
    merged = merged.merge(cpt, on=["Member_Key","YEAR"], how="left")
    merged = merged.merge(drg, on=["Member_Key","YEAR"], how="left")
    merged = merged.merge(icd, on=["Member_Key","YEAR"], how="left")

    agg_fill = [c for c in merged.columns if c.startswith(("cpt_","drg_","icd_"))]
    merged[agg_fill] = merged[agg_fill].fillna(0)
    print(f"    merged  → {merged.shape[0]:,} rows × {merged.shape[1]} cols")
    return merged


# ═══════════════════════════════════════════════════════════════════════════════
#  MAIN PIPELINE  — returns X, y_reg, y_cls, skf_splits, merged
# ═══════════════════════════════════════════════════════════════════════════════
def run_pipeline(main_raw, cpt_raw, drg_raw, icd_raw, dob_raw,
                 cost_threshold=COST_THRESHOLD, n_folds=N_FOLDS):
    """
    Process all training data.  No temporal split.
    Returns
      X          - feature matrix (DataFrame)
      y_reg      - TotalCost (regression target)
      y_cls      - binary HighCost label  (TotalCost > cost_threshold)
      skf_splits - list of (train_idx, val_idx) tuples from StratifiedKFold
      merged     - full merged DataFrame (for EDA)
    """
    print("\n" + "="*55)
    print("  Processing full training set")
    print("="*55)

    main_c = process_main(main_raw)
    dob_c  = process_dob(dob_raw)
    cpt_a  = aggregate_cpt(cpt_raw)
    drg_a  = aggregate_drg(drg_raw)
    icd_a  = aggregate_icd(icd_raw)

    merged = build_feature_table(main_c, dob_c, cpt_a, drg_a, icd_a)

    # Target
    y_reg = merged["TotalCost"].clip(lower=0).reset_index(drop=True)
    y_cls = (y_reg > cost_threshold).astype(int)

    # Feature matrix
    DROP = ["Member_Key", "YEAR", "TotalCost"]
    X = merged.drop(columns=[c for c in DROP if c in merged.columns]).copy()
    non_num = X.select_dtypes(exclude=[np.number]).columns.tolist()
    if non_num:
        print(f"    [WARN] dropping non-numeric: {non_num}")
        X.drop(columns=non_num, inplace=True)

    # ── Stratified K-Fold splits ─────────────────────────────────────────────
    skf        = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    skf_splits = list(skf.split(X, y_cls))     # list of (train_idx, val_idx)

    # Quick class report
    pos_rate = y_cls.mean()
    print(f"\n  ✓  X          : {X.shape}")
    print(f"  ✓  y_reg      : min={y_reg.min():.0f}  median={y_reg.median():.0f}  max={y_reg.max():,.0f}")
    print(f"  ✓  y_cls      : {y_cls.sum():,} positive / {len(y_cls):,} total  ({pos_rate*100:.1f}%)")
    print(f"  ✓  SKF folds  : {n_folds}  (each val fold ≈ {len(y_cls)//n_folds:,} rows)")
    print()

    # Fold-level class balance check
    print("  Fold  |  Train+  Train-  |  Val+   Val-   |  Val pos%")
    print("  ─────────────────────────────────────────────────────")
    for i, (tr, va) in enumerate(skf_splits):
        tr_pos = y_cls.iloc[tr].sum(); tr_neg = len(tr) - tr_pos
        va_pos = y_cls.iloc[va].sum(); va_neg = len(va) - va_pos
        print(f"  {i+1}     |  {tr_pos:6,}  {tr_neg:6,}  |  {va_pos:5,}  {va_neg:6,}  |  {va_pos/len(va)*100:.1f}%")

    return X, y_reg, y_cls, skf_splits, merged


# ═══════════════════════════════════════════════════════════════════════════════
#  ENTRY POINT
# ═══════════════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    import os

    print("[1/3] Loading raw files …")
    main_df = pd.read_csv(TRAINING_PATH + "main_df_train.csv", low_memory=False)
    cpt_df  = pd.read_csv(TRAINING_PATH + "cpt_df_train.csv",  low_memory=False)
    drg_df  = pd.read_csv(TRAINING_PATH + "drg_df_train.csv",  low_memory=False)
    icd_df  = pd.read_csv(TRAINING_PATH + "icd_df_train.csv",  low_memory=False)
    dob_df  = pd.read_csv(TRAINING_PATH + "dob_df.csv",         low_memory=False)

    print("[2/3] Running pipeline …")
    X, y_reg, y_cls, skf_splits, merged = run_pipeline(
        main_df, cpt_df, drg_df, icd_df, dob_df
    )

    print("[3/3] Quick sanity check on fold 1 …")
    tr_idx, va_idx = skf_splits[0]
    X_tr, X_va   = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va   = y_cls.iloc[tr_idx], y_cls.iloc[va_idx]
    print(f"  Fold 1 train: {X_tr.shape}  |  val: {X_va.shape}")
    print(f"  Train pos%: {y_tr.mean()*100:.1f}%   Val pos%: {y_va.mean()*100:.1f}%")
    print("\n[Done] preprocess.py complete.")

[1/3] Loading raw files …
[2/3] Running pipeline …

  Processing full training set
    main_df → 45,394 rows × 293 cols
    dob_df  → 64,443 rows × 4 cols
    cpt_df  → 162,526 rows after aggregation
    drg_df  → 230,814 rows after aggregation
    icd_df  → 162,525 rows after aggregation
    merged  → 45,394 rows × 309 cols

  ✓  X          : (45394, 306)
  ✓  y_reg      : min=524  median=3887  max=922,005
  ✓  y_cls      : 4,058 positive / 45,394 total  (8.9%)
  ✓  SKF folds  : 5  (each val fold ≈ 9,078 rows)

  Fold  |  Train+  Train-  |  Val+   Val-   |  Val pos%
  ─────────────────────────────────────────────────────
  1     |   3,247  33,068  |    811   8,268  |  8.9%
  2     |   3,246  33,069  |    812   8,267  |  8.9%
  3     |   3,246  33,069  |    812   8,267  |  8.9%
  4     |   3,246  33,069  |    812   8,267  |  8.9%
  5     |   3,247  33,069  |    811   8,267  |  8.9%
[3/3] Quick sanity check on fold 1 …
  Fold 1 train: (36315, 306)  |  val: (9079, 306)
  Train pos%: 8.9%

# **EDA**

In [1]:
"""
Healthcare Cost Prediction — Exploratory Data Analysis  v2
===========================================================
Fixes vs v1
  ✓ STANDALONE block now correctly calls new run_pipeline() (returns 5 values)
  ✓ Uses merged (not merged_train) and correct variable aliases
  ✓ TotalCost read from merged, not from dropped X
  ✓ y_cls used for HighCost labels (consistent with $30k threshold from preprocess)
  ✓ hcc_cols defined at top-level so section 9 can reference it
  ✓ Stale X_train / y_train / split_label references removed
  ✓ plt.sca() / plt.title() replaced with proper ax.set_title()
  ✓ Section guards (if col in df) added throughout to avoid KeyErrors

Sections
  1.  Data loading
  2.  Cost distribution   (histogram, log1p, Q-Q, percentiles, ECDF)
  3.  Outlier detection   (IQR fence, spend share, correlated cost cols)
  4.  Class imbalance     ($30k threshold + p80/p90 sweeps)
  5.  Demographic analysis (age, gender, age-group)
  6.  Clinical / disease features
  7.  HCC score analysis
  8.  Utilisation ↔ cost  (hexbin, violin)
  9.  Correlation heatmap
  10. Cohort / risk-tier analysis
  11. High-risk population segments
  12. Summary CSV
"""

# ─── 0. Imports ──────────────────────────────────────────────────────────────
import os, warnings, importlib.util
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr

warnings.filterwarnings("ignore")
matplotlib.rcParams.update({
    "figure.dpi"       : 130,
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "font.family"      : "DejaVu Sans",
    "axes.titlesize"   : 12,
    "axes.labelsize"   : 10,
})

# ─── Palette ─────────────────────────────────────────────────────────────────
C = dict(
    low     = "#4C9BE8",
    high    = "#E8654C",
    mid     = "#6DC78A",
    neutral = "#9B9EBB",
    accent  = "#F5A623",
    bg      = "#F7F8FC",
    grid    = "#E4E7F0",
)
PALETTE = [C["low"], C["mid"], C["high"], C["accent"], C["neutral"]]
sns.set_palette(PALETTE)

OUT = "./eda_plots"
os.makedirs(OUT, exist_ok=True)

def save(fig, name):
    fig.savefig(os.path.join(OUT, name), bbox_inches="tight", facecolor=C["bg"])
    print(f"    saved → {name}")
    plt.close(fig)


# ─────────────────────────────────────────────────────────────────────────────
#  1.  DATA LOADING
#      run_pipeline() returns: X, y_reg, y_cls, skf_splits, merged
# ─────────────────────────────────────────────────────────────────────────────
BASE_DIR   = r"/home/noneo/Codes/ML/MachineLearning/Hand_On_ML/Data/Healthcare Cost Dataset"
TRAIN_PATH = os.path.join(BASE_DIR, "train", "train") + os.sep

# ── Load preprocess module ────────────────────────────────────────────────────
_spec = importlib.util.spec_from_file_location("preprocess", "./preprocess.py")
pp    = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(pp)

# ── Load raw CSVs ─────────────────────────────────────────────────────────────
print("[1/12] Loading raw files …")
main_df = pd.read_csv(TRAIN_PATH + "main_df_train.csv", low_memory=False)
cpt_df  = pd.read_csv(TRAIN_PATH + "cpt_df_train.csv",  low_memory=False)
drg_df  = pd.read_csv(TRAIN_PATH + "drg_df_train.csv",  low_memory=False)
icd_df  = pd.read_csv(TRAIN_PATH + "icd_df_train.csv",  low_memory=False)
dob_df  = pd.read_csv(TRAIN_PATH + "dob_df.csv",         low_memory=False)

# ── Run preprocessing pipeline (ALL training data, no temporal split) ─────────
X, y_reg, y_cls, skf_splits, merged = pp.run_pipeline(
    main_df, cpt_df, drg_df, icd_df, dob_df
)

# ── Convenience aliases ───────────────────────────────────────────────────────
df   = merged.copy()                          # full merged table including TotalCost
cost = df["TotalCost"].clip(lower=0)          # regression target
log_cost = np.log1p(cost)

# Binary label consistent with model_pipeline ($30k threshold)
COST_THRESHOLD   = pp.COST_THRESHOLD          # 30_000
df["HighCost"]   = y_cls.values               # 1 = TotalCost > $30k
df["HighCost_p80"] = (cost > cost.quantile(0.80)).astype(int)
df["HighCost_p90"] = (cost > cost.quantile(0.90)).astype(int)

# Pre-declare hcc_cols so section 9 can reference them even if section 7 skips
hcc_cols = [c for c in ["MemberHccScore","MemberHccScoreLastYear",
                          "MemberHccScoreLastTwoYear","PersiviaMemberHccScore"]
            if c in df.columns]

print(f"\n[EDA] {df.shape[0]:,} members × {df.shape[1]} features")
print(f"      TotalCost — min ${cost.min():,.0f}  "
      f"median ${cost.median():,.0f}  max ${cost.max():,.0f}  mean ${cost.mean():,.0f}")
print(f"      HighCost (>${COST_THRESHOLD:,}): {df['HighCost'].sum():,} "
      f"({df['HighCost'].mean()*100:.1f}%)")


# ─────────────────────────────────────────────────────────────────────────────
#  2.  COST DISTRIBUTION
# ─────────────────────────────────────────────────────────────────────────────
print("\n[2/12] Cost distribution …")

fig = plt.figure(figsize=(16, 10), facecolor=C["bg"])
fig.suptitle("Healthcare Cost Distribution", fontsize=15, fontweight="bold", y=1.01)
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

# 2a. Raw histogram
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(cost, bins=120, color=C["low"], edgecolor="none", alpha=0.85)
ax1.set_title("Raw TotalCost")
ax1.set_xlabel("Cost ($)")
ax1.set_ylabel("Members")
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}k"))
ax1.axvline(cost.mean(),   color=C["high"],   lw=1.5, ls="--",
            label=f"Mean  ${cost.mean():,.0f}")
ax1.axvline(cost.median(), color=C["accent"], lw=1.5, ls=":",
            label=f"Median ${cost.median():,.0f}")
ax1.axvline(COST_THRESHOLD, color="black", lw=1.2, ls="-.",
            label=f"HighCost thr ${COST_THRESHOLD:,}")
ax1.legend(fontsize=7)

# 2b. log1p histogram
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(log_cost, bins=100, color=C["mid"], edgecolor="none", alpha=0.85)
ax2.set_title("log1p(TotalCost)")
ax2.set_xlabel("log1p(Cost)")
ax2.set_ylabel("Members")
ax2.axvline(np.log1p(COST_THRESHOLD), color="black", lw=1.2, ls="-.",
            label=f"log1p({COST_THRESHOLD:,})")
ax2.legend(fontsize=7)

# 2c. Normal Q-Q plot on log1p
ax3 = fig.add_subplot(gs[0, 2])
(osm, osr), (slope, intercept, r) = stats.probplot(log_cost, dist="norm")
ax3.scatter(osm, osr, s=4, alpha=0.4, color=C["neutral"])
ax3.plot(osm, slope * np.array(osm) + intercept, color=C["high"], lw=1.5)
ax3.set_title(f"Q-Q Plot (log1p)   R²={r**2:.3f}")
ax3.set_xlabel("Theoretical quantiles")
ax3.set_ylabel("Sample quantiles")

# 2d. Percentile bar
pcts = [50, 75, 90, 95, 99, 99.5]
vals = [np.percentile(cost, p) for p in pcts]
ax4  = fig.add_subplot(gs[1, 0])
bars = ax4.barh([f"p{p}" for p in pcts], vals,
                color=[C["low"], C["low"], C["mid"], C["mid"], C["high"], C["high"]])
ax4.set_xlabel("Cost ($)")
ax4.set_title("Cost Percentiles")
ax4.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}k"))
for bar, v in zip(bars, vals):
    ax4.text(bar.get_width()*1.01, bar.get_y()+bar.get_height()/2,
             f"${v:,.0f}", va="center", fontsize=8)

# 2e. Stats panel
sk_raw = stats.skew(cost);     ku_raw = stats.kurtosis(cost)
sk_log = stats.skew(log_cost); ku_log = stats.kurtosis(log_cost)
ax5 = fig.add_subplot(gs[1, 1])
ax5.axis("off")
txt = (
    f"{'Metric':<22} {'Raw':>12} {'log1p':>12}\n"
    f"{'─'*46}\n"
    f"{'Skewness':<22} {sk_raw:>12.3f} {sk_log:>12.3f}\n"
    f"{'Kurtosis (excess)':<22} {ku_raw:>12.3f} {ku_log:>12.3f}\n"
    f"{'Mean':<22} {cost.mean():>12,.0f}\n"
    f"{'Median':<22} {cost.median():>12,.0f}\n"
    f"{'Std Dev':<22} {cost.std():>12,.0f}\n"
    f"{'CV (std/mean)':<22} {cost.std()/cost.mean():>12.3f}\n"
    f"{'Zero-cost members':<22} {(cost==0).sum():>12,}\n"
    f"{'HighCost members':<22} {df['HighCost'].sum():>12,}\n"
)
ax5.text(0.02, 0.98, txt, transform=ax5.transAxes, fontsize=9, va="top",
         fontfamily="monospace",
         bbox=dict(boxstyle="round,pad=0.5", fc=C["grid"], ec="none"))
ax5.set_title("Descriptive Statistics")

# 2f. ECDF
ax6 = fig.add_subplot(gs[1, 2])
sorted_cost = np.sort(cost)
cdf = np.arange(1, len(sorted_cost)+1) / len(sorted_cost)
ax6.plot(sorted_cost, cdf, color=C["low"], lw=1.5)
ax6.axhline(0.90, color=C["accent"], lw=1, ls="--", label="p90")
ax6.axhline(0.99, color=C["high"],   lw=1, ls="--", label="p99")
ax6.axvline(COST_THRESHOLD, color="black", lw=1.2, ls="-.",
            label=f"${COST_THRESHOLD/1e3:.0f}k")
ax6.set_title("ECDF of TotalCost")
ax6.set_xlabel("Cost ($)")
ax6.set_ylabel("Cumulative fraction")
ax6.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}k"))
ax6.legend(fontsize=8)

save(fig, "02_cost_distribution.png")


# ─────────────────────────────────────────────────────────────────────────────
#  3.  OUTLIER DETECTION
# ─────────────────────────────────────────────────────────────────────────────
print("[3/12] Outlier detection …")

Q1, Q3     = cost.quantile(0.25), cost.quantile(0.75)
IQR        = Q3 - Q1
upper_fence = Q3 + 3.0 * IQR
n_outliers  = (cost > upper_fence).sum()
pct_spend_outliers = cost[cost > upper_fence].sum() / cost.sum() * 100

fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor=C["bg"])
fig.suptitle("Outlier Analysis", fontsize=14, fontweight="bold")

# 3a. Boxplot log scale
axes[0].boxplot(cost, vert=True, patch_artist=True,
                boxprops    =dict(facecolor=C["low"],  color=C["neutral"]),
                medianprops =dict(color=C["high"], lw=2),
                flierprops  =dict(marker=".", ms=3, alpha=0.3, color=C["high"]),
                whiskerprops=dict(color=C["neutral"]),
                capprops    =dict(color=C["neutral"]))
axes[0].set_yscale("log")
axes[0].set_title("Boxplot (log scale)")
axes[0].set_ylabel("TotalCost ($)")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f"${y:,.0f}"))

# 3b. Spend share pie
labels = [
    f"Top 1%\n({(cost>cost.quantile(0.99)).sum():,} members)",
    "Next 9%", "Bottom 90%",
]
sizes = [
    cost[cost > cost.quantile(0.99)].sum(),
    cost[(cost > cost.quantile(0.90)) & (cost <= cost.quantile(0.99))].sum(),
    cost[cost <= cost.quantile(0.90)].sum(),
]
axes[1].pie(sizes, labels=labels,
            colors=[C["high"], C["accent"], C["low"]],
            autopct="%1.1f%%", startangle=140,
            wedgeprops=dict(edgecolor="white", linewidth=1.5))
axes[1].set_title("Share of Total Healthcare Spend")

# 3c. Cost sub-columns most correlated with outlier flag
outlier_flag = (cost > upper_fence).astype(int)
cost_cols    = [c for c in df.columns if c.endswith("_Cost") and c != "TotalCost"]
corrs = {}
for c in cost_cols:
    col_data = df[c].fillna(0)
    if col_data.nunique() > 1:
        corrs[c] = pearsonr(col_data, outlier_flag)[0]
top_c  = sorted(corrs, key=lambda x: abs(corrs[x]), reverse=True)[:12]
vals_c = [corrs[k] for k in top_c]
axes[2].barh(top_c[::-1], vals_c[::-1],
             color=[C["high"] if v > 0 else C["low"] for v in vals_c[::-1]])
axes[2].axvline(0, color=C["neutral"], lw=0.8)
axes[2].set_title("Cost Features vs Outlier Flag\n(Pearson r)")
axes[2].set_xlabel("Correlation")

fig.text(0.5, -0.02,
         f"Tukey 3×IQR fence = ${upper_fence:,.0f}  |  "
         f"Outliers: {n_outliers:,} ({n_outliers/len(cost)*100:.1f}%)  |  "
         f"Drive {pct_spend_outliers:.1f}% of total spend",
         ha="center", fontsize=9, color=C["neutral"])
plt.tight_layout()
save(fig, "03_outlier_analysis.png")


# ─────────────────────────────────────────────────────────────────────────────
#  4.  CLASS IMBALANCE
# ─────────────────────────────────────────────────────────────────────────────
print("[4/12] Class imbalance …")

# Use the model's actual $30k threshold AND percentile-based views
hc_30k  = df["HighCost"]
n_high  = hc_30k.sum()
n_low   = len(hc_30k) - n_high
s_high  = cost[hc_30k == 1].sum()
s_low   = cost[hc_30k == 0].sum()

fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor=C["bg"])
fig.suptitle(f"Class Imbalance  (HighCost threshold = ${COST_THRESHOLD:,})",
             fontsize=13, fontweight="bold")

# 4a. Member count
axes[0].bar(["Low-Cost","High-Cost"], [n_low, n_high],
            color=[C["low"], C["high"]], width=0.5, edgecolor="white", linewidth=1.5)
for i, v in enumerate([n_low, n_high]):
    axes[0].text(i, v + 30, f"{v:,}\n({v/len(hc_30k)*100:.1f}%)",
                 ha="center", fontsize=10, fontweight="bold")
axes[0].set_title("Member Count")
axes[0].set_ylabel("Members")

# 4b. Spend concentration
axes[1].bar(["Low-Cost","High-Cost"], [s_low, s_high],
            color=[C["low"], C["high"]], width=0.5, edgecolor="white")
for i, v in enumerate([s_low, s_high]):
    axes[1].text(i, v*1.01, f"${v/1e6:.1f}M\n({v/(s_low+s_high)*100:.0f}%)",
                 ha="center", fontsize=10)
axes[1].set_title("Total Spend")
axes[1].set_ylabel("Total Cost ($)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e6:.1f}M"))

# 4c. Multiple threshold sweep
thresholds  = [50, 60, 70, 75, 80, 85, 90, 95]
pct_members = [(cost > cost.quantile(p/100)).mean()*100 for p in thresholds]
pct_spend   = [cost[cost > cost.quantile(p/100)].sum()/cost.sum()*100 for p in thresholds]
x = np.arange(len(thresholds)); w = 0.35
axes[2].bar(x - w/2, pct_members, w, label="% Members",    color=C["low"])
axes[2].bar(x + w/2, pct_spend,   w, label="% Total Spend", color=C["high"])
axes[2].set_xticks(x)
axes[2].set_xticklabels([f"p{p}" for p in thresholds], fontsize=8)
axes[2].set_title("Members vs Spend by Percentile Threshold")
axes[2].set_ylabel("%")
axes[2].legend(fontsize=8)
axes[2].axhline(50, color=C["neutral"], lw=0.8, ls="--")

# Mark model threshold on bar chart
thr_pos_rate = hc_30k.mean() * 100
axes[2].text(0.02, 0.96, f"Model thr: {thr_pos_rate:.1f}% positive",
             transform=axes[2].transAxes, fontsize=8, va="top", color=C["high"])
plt.tight_layout()
save(fig, "04_class_imbalance.png")


# ─────────────────────────────────────────────────────────────────────────────
#  5.  DEMOGRAPHIC ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────
print("[5/12] Demographics …")

fig, axes = plt.subplots(2, 3, figsize=(16, 10), facecolor=C["bg"])
fig.suptitle("Demographic Analysis", fontsize=14, fontweight="bold")

# 5a. Age distribution
if "Age" in df.columns:
    axes[0,0].hist(df["Age"].dropna(), bins=40,
                   color=C["low"], edgecolor="none", alpha=0.85)
    axes[0,0].axvline(df["Age"].median(), color=C["high"], lw=1.5, ls="--",
                      label=f"Median {df['Age'].median():.0f}")
    axes[0,0].set_title("Age Distribution")
    axes[0,0].set_xlabel("Age"); axes[0,0].set_ylabel("Members")
    axes[0,0].legend(fontsize=8)
else:
    axes[0,0].axis("off"); axes[0,0].set_title("Age — not available")

# 5b. Average cost by age band
if "Age" in df.columns:
    df["AgeBand"] = pd.cut(df["Age"],
                           bins=[0,45,55,65,70,75,80,85,120],
                           labels=["<45","45-54","55-64","65-69",
                                   "70-74","75-79","80-84","85+"])
    age_cost = df.groupby("AgeBand", observed=True)["TotalCost"].mean().reset_index()
    axes[0,1].bar(age_cost["AgeBand"].astype(str), age_cost["TotalCost"],
                  color=PALETTE[:len(age_cost)])
    axes[0,1].set_title("Mean Cost by Age Band")
    axes[0,1].set_xlabel("Age Band"); axes[0,1].set_ylabel("Mean TotalCost ($)")
    axes[0,1].tick_params(axis="x", rotation=30)
    axes[0,1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
else:
    axes[0,1].axis("off")

# 5c. High-cost rate by age band  (uses $30k label)
if "Age" in df.columns:
    hc_age = df.groupby("AgeBand", observed=True)["HighCost"].mean().reset_index()
    axes[0,2].bar(hc_age["AgeBand"].astype(str), hc_age["HighCost"]*100,
                  color=PALETTE[:len(hc_age)])
    axes[0,2].axhline(hc_30k.mean()*100, color=C["high"], lw=1, ls="--",
                      label=f"Overall {hc_30k.mean()*100:.1f}%")
    axes[0,2].set_title("High-Cost Rate by Age Band")
    axes[0,2].set_xlabel("Age Band"); axes[0,2].set_ylabel("% High-Cost Members")
    axes[0,2].tick_params(axis="x", rotation=30)
    axes[0,2].legend(fontsize=8)
else:
    axes[0,2].axis("off")

# 5d. Gender count
if "Gender_Key" in df.columns:
    g_map    = {1:"Male", 2:"Female", 0:"Unknown"}
    g_counts = df["Gender_Key"].map(g_map).fillna("Other").value_counts()
    axes[1,0].bar(g_counts.index.astype(str), g_counts.values,
                  color=[C["low"], C["high"], C["neutral"]][:len(g_counts)])
    axes[1,0].set_title("Gender Distribution"); axes[1,0].set_ylabel("Members")
    for i, v in enumerate(g_counts.values):
        axes[1,0].text(i, v + 5, f"{v:,}", ha="center", fontsize=9)
    df["Gender_Label"] = df["Gender_Key"].map(g_map).fillna("Other")
else:
    axes[1,0].axis("off")
    df["Gender_Label"] = "Unknown"

# 5e. Cost boxplot by gender (no plt.sca — use ax directly)
if "Gender_Label" in df.columns and df["Gender_Label"].nunique() > 1:
    groups = [grp["TotalCost"].values
              for _, grp in df.groupby("Gender_Label")]
    g_labels = df["Gender_Label"].unique().tolist()
    axes[1,1].boxplot(groups, labels=g_labels, patch_artist=True,
                      boxprops=dict(facecolor=C["low"]),
                      medianprops=dict(color=C["high"], lw=2),
                      flierprops=dict(marker=".", ms=2, alpha=0.3))
    axes[1,1].set_yscale("log")
    axes[1,1].set_title("Cost Distribution by Gender")
    axes[1,1].set_ylabel("TotalCost (log scale)")
    axes[1,1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f"${y:,.0f}"))
else:
    axes[1,1].axis("off")

# 5f. AgeGroup_Key high-cost rate
if "AgeGroup_Key" in df.columns:
    ag_hc = df.groupby("AgeGroup_Key", observed=True)["HighCost"].mean().reset_index()
    axes[1,2].bar(ag_hc["AgeGroup_Key"].astype(str), ag_hc["HighCost"]*100, color=C["low"])
    axes[1,2].set_title("High-Cost Rate by AgeGroup_Key")
    axes[1,2].set_xlabel("AgeGroup_Key"); axes[1,2].set_ylabel("% High-Cost Members")
else:
    axes[1,2].axis("off")

plt.tight_layout()
save(fig, "05_demographics.png")


# ─────────────────────────────────────────────────────────────────────────────
#  6.  CLINICAL / DISEASE FEATURES
# ─────────────────────────────────────────────────────────────────────────────
print("[6/12] Disease features …")

disease_cols = [c for c in [
    "ChronicObstructivePulmonaryDiseaseOrAsthma", "CongestiveHeartFailure",
    "BacterialPneumonia", "DiabetesShortTermComplications",
    "DiabetesLongTermComplications", "UncontrolledDiabetes",
    "AmputationDiabetes", "Dialysis", "ChronicConditions",
] if c in df.columns]

short_names = {
    "ChronicObstructivePulmonaryDiseaseOrAsthma": "COPD/Asthma",
    "CongestiveHeartFailure":                     "CHF",
    "BacterialPneumonia":                         "Pneumonia",
    "DiabetesShortTermComplications":             "Diabetes (ST)",
    "DiabetesLongTermComplications":              "Diabetes (LT)",
    "UncontrolledDiabetes":                       "Uncontrolled DM",
    "AmputationDiabetes":                         "DM Amputation",
    "Dialysis":                                   "Dialysis",
    "ChronicConditions":                          "Chronic Conds (count)",
}

fig, axes = plt.subplots(1, 3, figsize=(17, 6), facecolor=C["bg"])
fig.suptitle("Clinical / Disease Feature Analysis", fontsize=14, fontweight="bold")

# 6a. Prevalence
prev = {short_names.get(c, c): (df[c] > 0).mean()*100
        for c in disease_cols}
prev = dict(sorted(prev.items(), key=lambda x: x[1]))
axes[0].barh(list(prev.keys()), list(prev.values()), color=C["low"])
axes[0].set_xlabel("Prevalence (%)")
axes[0].set_title("Disease Prevalence in Cohort")
for i, (k, v) in enumerate(prev.items()):
    axes[0].text(v + 0.15, i, f"{v:.1f}%", va="center", fontsize=8)

# 6b. Mean cost: with vs without (binary disease flags only)
bin_diseases = [c for c in disease_cols
                if df[c].nunique() <= 5 and c != "ChronicConditions"]
if bin_diseases:
    cost_with    = {short_names.get(c,c): df.loc[df[c]>0, "TotalCost"].mean()
                    for c in bin_diseases}
    cost_without = {short_names.get(c,c): df.loc[df[c]==0, "TotalCost"].mean()
                    for c in bin_diseases}
    keys = list(cost_with.keys())
    x    = np.arange(len(keys)); w = 0.38
    axes[1].bar(x-w/2, [cost_with[k]    for k in keys], w,
                label="With condition",    color=C["high"], alpha=0.85)
    axes[1].bar(x+w/2, [cost_without[k] for k in keys], w,
                label="Without condition", color=C["low"],  alpha=0.85)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(keys, rotation=35, ha="right", fontsize=8)
    axes[1].set_title("Mean Cost: With vs Without Condition")
    axes[1].set_ylabel("Mean TotalCost ($)")
    axes[1].legend(fontsize=8)
    axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
else:
    axes[1].axis("off")

# 6c. High-cost rate: with vs without ($30k label)
if bin_diseases:
    hc_with    = {short_names.get(c,c): df.loc[df[c]>0, "HighCost"].mean()*100
                  for c in bin_diseases}
    hc_without = {short_names.get(c,c): df.loc[df[c]==0, "HighCost"].mean()*100
                  for c in bin_diseases}
    y_with    = [hc_with[k]    for k in keys]
    y_without = [hc_without[k] for k in keys]
    y_pos = np.arange(len(keys))
    axes[2].barh(y_pos + 0.2, y_with,    0.38, color=C["high"], alpha=0.85, label="With")
    axes[2].barh(y_pos - 0.2, y_without, 0.38, color=C["low"],  alpha=0.85, label="Without")
    axes[2].set_yticks(y_pos)
    axes[2].set_yticklabels(keys, fontsize=8)
    axes[2].set_xlabel("High-Cost Rate (%)")
    axes[2].set_title("High-Cost Rate by Condition")
    axes[2].legend(fontsize=8)
else:
    axes[2].axis("off")

plt.tight_layout()
save(fig, "06_disease_features.png")


# ─────────────────────────────────────────────────────────────────────────────
#  7.  HCC SCORE ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────
print("[7/12] HCC scores …")

fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor=C["bg"])
fig.suptitle("HCC Score Analysis", fontsize=14, fontweight="bold")

if "MemberHccScore" in df.columns:
    # 7a. Distribution
    axes[0].hist(df["MemberHccScore"].clip(0, 5), bins=50,
                 color=C["neutral"], edgecolor="none", alpha=0.85)
    axes[0].set_title("HCC Score Distribution")
    axes[0].set_xlabel("MemberHccScore"); axes[0].set_ylabel("Members")

    # 7b. Hexbin: HCC vs log1p(cost)
    hb = axes[1].hexbin(df["MemberHccScore"].clip(0, 5), log_cost,
                        gridsize=40, cmap="YlOrRd", mincnt=1)
    axes[1].set_title("HCC Score vs log1p(Cost)")
    axes[1].set_xlabel("MemberHccScore"); axes[1].set_ylabel("log1p(TotalCost)")
    fig.colorbar(hb, ax=axes[1], label="Count")

    # 7c. YoY HCC delta
    if "MemberHccScoreLastYear" in df.columns:
        df["HCC_Delta"] = df["MemberHccScore"] - df["MemberHccScoreLastYear"]
        axes[2].hist(df["HCC_Delta"].clip(-3, 3), bins=60,
                     color=C["accent"], edgecolor="none", alpha=0.85)
        axes[2].axvline(0, color=C["high"], lw=1.5, ls="--")
        pct_worse = (df["HCC_Delta"] > 0).mean() * 100
        axes[2].text(0.98, 0.96, f"{pct_worse:.1f}% worsened",
                     transform=axes[2].transAxes, ha="right", va="top",
                     fontsize=9, color=C["high"])
        axes[2].set_title("YoY HCC Score Change")
        axes[2].set_xlabel("Δ HCC (current − last year)")
        axes[2].set_ylabel("Members")
    else:
        axes[2].axis("off"); axes[2].set_title("HCC LastYear — not available")
else:
    for ax in axes:
        ax.axis("off"); ax.set_title("MemberHccScore — not available")

plt.tight_layout()
save(fig, "07_hcc_scores.png")


# ─────────────────────────────────────────────────────────────────────────────
#  8.  UTILISATION ↔ COST RELATIONSHIPS
# ─────────────────────────────────────────────────────────────────────────────
print("[8/12] Utilisation vs cost …")

util_map = {
    "ProviderVisitCount"           : "Provider Visits",
    "EmergencyDepartmentVisits"    : "ED Visits",
    "Inpatient"                    : "Inpatient Admits",
    "AdmissionLOS"                 : "Admission LOS",
    "PCPVisits"                    : "PCP Visits",
    "TotalScriptsFilled"           : "Scripts Filled",
    "ChronicConditions"            : "Chronic Conditions",
}
util_cols = {k: v for k, v in util_map.items() if k in df.columns}

fig, axes = plt.subplots(2, 4, figsize=(18, 9), facecolor=C["bg"])
fig.suptitle("Utilisation Patterns vs Future Cost", fontsize=14, fontweight="bold")
axes = axes.flatten()

for idx, (col, label) in enumerate(util_cols.items()):
    ax = axes[idx]
    x  = df[col].clip(upper=df[col].quantile(0.99)).fillna(0)
    hb = ax.hexbin(x, log_cost, gridsize=35, cmap="Blues", mincnt=1)
    r, _ = spearmanr(x, log_cost)
    ax.set_title(f"{label}\nSpearman r={r:.3f}", fontsize=9)
    ax.set_xlabel(label, fontsize=8)
    ax.set_ylabel("log1p(Cost)", fontsize=8)
    fig.colorbar(hb, ax=ax)

# Violin: ED visit bands vs cost
ax = axes[len(util_cols)]
if "EmergencyDepartmentVisits" in df.columns:
    df["ED_Band"] = df["EmergencyDepartmentVisits"].clip(0, 3).astype(int).astype(str)
    df.loc[df["EmergencyDepartmentVisits"] >= 3, "ED_Band"] = "3+"
    order = [o for o in ["0","1","2","3+"] if o in df["ED_Band"].unique()]
    sns.violinplot(data=df, x="ED_Band", y="TotalCost", order=order,
                   palette=[C["low"], C["mid"], C["accent"], C["high"]],
                   ax=ax, inner="quartile", cut=0)
    ax.set_yscale("log")
    ax.set_title("Cost by ED Visit Count")
    ax.set_xlabel("ED Visits"); ax.set_ylabel("TotalCost (log)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f"${y:,.0f}"))
else:
    ax.axis("off")

for i in range(len(util_cols)+1, len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()
save(fig, "08_utilisation_cost.png")


# ─────────────────────────────────────────────────────────────────────────────
#  9.  CORRELATION HEATMAP
# ─────────────────────────────────────────────────────────────────────────────
print("[9/12] Correlation heatmap …")

candidate_cols = list(dict.fromkeys(
    [c for c in df.columns if c.endswith("_Cost") and c != "TotalCost"][:12]
    + list(util_cols.keys())
    + hcc_cols                        # defined at top of script — always safe
    + [c for c in disease_cols[:6] if c in df.columns]
    + ["TotalCost"]
))
# Keep only numeric columns that actually exist
candidate_cols = [c for c in candidate_cols if c in df.columns
                  and pd.api.types.is_numeric_dtype(df[c])]
corr_df = df[candidate_cols].fillna(0).corr()

short = {}
for c in corr_df.columns:
    s = c
    for old, new in [
        ("_Cost","_C"), ("EmergencyDepartment","ED"),
        ("ChronicObstructivePulmonaryDiseaseOrAsthma","COPD"),
        ("CongestiveHeartFailure","CHF"),
        ("DiabetesShortTermComplications","DM_ST"),
        ("DiabetesLongTermComplications","DM_LT"),
        ("PrimaryCareServicesTotal","PCS_Total"),
        ("MemberHccScore","HCC"), ("MemberHccScoreLastYear","HCC_1yr"),
        ("PersiviaMemberHccScore","PersiviaHCC"),
    ]:
        s = s.replace(old, new)
    short[c] = s
corr_df.rename(columns=short, index=short, inplace=True)

fig, ax = plt.subplots(figsize=(14, 12), facecolor=C["bg"])
mask = np.triu(np.ones_like(corr_df, dtype=bool), k=1)
sns.heatmap(corr_df, mask=mask, ax=ax,
            cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            linewidths=0.4, linecolor=C["bg"],
            annot=(len(corr_df) <= 22), fmt=".1f",
            annot_kws={"size": 7},
            cbar_kws={"shrink": 0.7})
ax.set_title("Feature Correlation Matrix (lower triangle)",
             fontsize=13, fontweight="bold", pad=12)
plt.tight_layout()
save(fig, "09_correlation_heatmap.png")


# ─────────────────────────────────────────────────────────────────────────────
#  10. COHORT / RISK TIER ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────
print("[10/12] Cohort / risk tier analysis …")

risk_features = [c for c in [
    "MemberHccScore","ChronicConditions","EmergencyDepartmentVisits",
    "Inpatient","AdmissionLOS",
] + (["HCC_Delta"] if "HCC_Delta" in df.columns else [])
if c in df.columns]

risk_raw  = df[risk_features].fillna(0)
risk_norm = (risk_raw - risk_raw.min()) / (risk_raw.max() - risk_raw.min() + 1e-9)
df["RiskScore"] = risk_norm.mean(axis=1)
df["RiskTier"]  = pd.qcut(df["RiskScore"], q=4,
                           labels=["Low","Moderate","High","Very High"])

tier_colors = [C["low"], C["mid"], C["accent"], C["high"]]
tier_order  = ["Low","Moderate","High","Very High"]

fig, axes = plt.subplots(1, 3, figsize=(16, 6), facecolor=C["bg"])
fig.suptitle("Risk Tier Cohort Analysis", fontsize=14, fontweight="bold")

tier_counts = df["RiskTier"].value_counts().reindex(tier_order)
axes[0].bar(tier_order, tier_counts.values, color=tier_colors, width=0.55)
axes[0].set_title("Member Count by Risk Tier"); axes[0].set_ylabel("Members")
for i, v in enumerate(tier_counts.values):
    axes[0].text(i, v + 5, f"{v:,}", ha="center", fontsize=9)

tier_mean   = df.groupby("RiskTier", observed=True)["TotalCost"].mean().reindex(tier_order)
tier_median = df.groupby("RiskTier", observed=True)["TotalCost"].median().reindex(tier_order)
x = np.arange(len(tier_order)); w = 0.38
axes[1].bar(x-w/2, tier_mean.values,   w, color=tier_colors, label="Mean")
axes[1].bar(x+w/2, tier_median.values, w, color=tier_colors, alpha=0.4,
            edgecolor="grey", label="Median")
axes[1].set_xticks(x); axes[1].set_xticklabels(tier_order)
axes[1].set_title("Mean & Median Cost per Tier")
axes[1].set_ylabel("TotalCost ($)"); axes[1].legend(fontsize=8)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))

tier_spend = df.groupby("RiskTier", observed=True)["TotalCost"].sum().reindex(tier_order)
axes[2].pie(tier_spend.values, labels=tier_order, colors=tier_colors,
            autopct="%1.1f%%", startangle=140,
            wedgeprops=dict(edgecolor="white", linewidth=1.5))
axes[2].set_title("Total Spend Share by Risk Tier")

plt.tight_layout()
save(fig, "10_risk_tier_cohort.png")


# ─────────────────────────────────────────────────────────────────────────────
#  11. HIGH-RISK POPULATION SEGMENTS
# ─────────────────────────────────────────────────────────────────────────────
print("[11/12] High-risk segments …")

vhr = df[df["RiskTier"] == "Very High"]
seg_features = [c for c in [
    "Age","MemberHccScore","ChronicConditions",
    "EmergencyDepartmentVisits","Inpatient","AdmissionLOS",
] if c in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(16, 10), facecolor=C["bg"])
fig.suptitle("High-Risk vs All Members — Feature Distributions",
             fontsize=14, fontweight="bold")
axes = axes.flatten()

for idx, feat in enumerate(seg_features[:6]):
    ax   = axes[idx]
    clip = df[feat].quantile(0.99)
    ax.hist(df[feat].clip(upper=clip).fillna(0),  bins=40, alpha=0.55,
            color=C["low"],  label="All",         density=True)
    ax.hist(vhr[feat].clip(upper=clip).fillna(0), bins=40, alpha=0.70,
            color=C["high"], label="Very High risk", density=True)
    ax.set_title(feat, fontsize=9); ax.set_ylabel("Density"); ax.legend(fontsize=7)

plt.tight_layout()
save(fig, "11a_high_risk_distributions.png")

# Multi-morbidity
cond_cols = [c for c in [
    "CongestiveHeartFailure","ChronicObstructivePulmonaryDiseaseOrAsthma",
    "DiabetesLongTermComplications","DiabetesShortTermComplications",
    "UncontrolledDiabetes","BacterialPneumonia","Dialysis",
] if c in df.columns]

if cond_cols:
    df["ConditionCount"] = df[cond_cols].gt(0).sum(axis=1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), facecolor=C["bg"])
    fig.suptitle("Multi-Morbidity Analysis", fontsize=13, fontweight="bold")
    cc_dist = df["ConditionCount"].value_counts().sort_index()
    axes[0].bar(cc_dist.index.astype(str), cc_dist.values, color=C["low"])
    axes[0].set_title("Members by # Conditions")
    axes[0].set_xlabel("# Conditions"); axes[0].set_ylabel("Members")
    cc_cost = df.groupby("ConditionCount")["TotalCost"].mean()
    axes[1].plot(cc_cost.index, cc_cost.values, marker="o", color=C["high"], lw=2, ms=6)
    axes[1].fill_between(cc_cost.index, cc_cost.values, alpha=0.15, color=C["high"])
    axes[1].set_title("Mean Cost by # Conditions")
    axes[1].set_xlabel("# Conditions"); axes[1].set_ylabel("Mean TotalCost ($)")
    axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
    plt.tight_layout()
    save(fig, "11b_multimorbidity.png")


# ─────────────────────────────────────────────────────────────────────────────
#  12. SUMMARY TABLE
# ─────────────────────────────────────────────────────────────────────────────
print("[12/12] Summary table …")

rows = []
rows += [
    ("TotalCost", "Mean ($)",         f"{cost.mean():,.0f}"),
    ("TotalCost", "Median ($)",       f"{cost.median():,.0f}"),
    ("TotalCost", "Std Dev ($)",      f"{cost.std():,.0f}"),
    ("TotalCost", "Skewness (raw)",   f"{sk_raw:.3f}"),
    ("TotalCost", "Kurtosis (raw)",   f"{ku_raw:.3f}"),
    ("TotalCost", "Skewness (log1p)", f"{sk_log:.3f}"),
    ("TotalCost", "p99 ($)",          f"{cost.quantile(0.99):,.0f}"),
]
rows += [
    ("Class ($30k)", "High-cost count",   f"{n_high:,}"),
    ("Class ($30k)", "Low-cost count",    f"{n_low:,}"),
    ("Class ($30k)", "Spend share",       f"{s_high/(s_low+s_high)*100:.1f}%"),
    ("Class ($30k)", "Imbalance ratio",   f"{n_low/max(n_high,1):.1f}:1"),
]
rows += [
    ("Outliers", "Tukey 3×IQR fence ($)", f"{upper_fence:,.0f}"),
    ("Outliers", "Outlier members",        f"{n_outliers:,}"),
    ("Outliers", "Outlier spend share",    f"{pct_spend_outliers:.1f}%"),
]
for tier in tier_order:
    n = (df["RiskTier"] == tier).sum()
    m = df.loc[df["RiskTier"] == tier, "TotalCost"].mean()
    rows.append(("RiskTier", f"{tier} — count",     f"{n:,}"))
    rows.append(("RiskTier", f"{tier} — mean cost", f"${m:,.0f}"))

summary_df = pd.DataFrame(rows, columns=["Category","Metric","Value"])
print("\n" + summary_df.to_string(index=False))
summary_df.to_csv(os.path.join(OUT, "eda_summary.csv"), index=False)

print(f"\n[EDA complete]  plots → {OUT}/")
print(f"Files: {sorted(os.listdir(OUT))}")

[1/12] Loading raw files …

  Processing full training set
    main_df → 45,394 rows × 293 cols
    dob_df  → 64,443 rows × 4 cols
    cpt_df  → 162,526 rows after aggregation
    drg_df  → 230,814 rows after aggregation
    icd_df  → 162,525 rows after aggregation
    merged  → 45,394 rows × 309 cols

  ✓  X          : (45394, 306)
  ✓  y_reg      : min=524  median=3887  max=922,005
  ✓  y_cls      : 4,058 positive / 45,394 total  (8.9%)
  ✓  SKF folds  : 5  (each val fold ≈ 9,078 rows)

  Fold  |  Train+  Train-  |  Val+   Val-   |  Val pos%
  ─────────────────────────────────────────────────────
  1     |   3,247  33,068  |    811   8,268  |  8.9%
  2     |   3,246  33,069  |    812   8,267  |  8.9%
  3     |   3,246  33,069  |    812   8,267  |  8.9%
  4     |   3,246  33,069  |    812   8,267  |  8.9%
  5     |   3,247  33,069  |    811   8,267  |  8.9%

[EDA] 45,394 members × 312 features
      TotalCost — min $524  median $3,887  max $922,005  mean $11,036
      HighCost (>$30,0

In [ ]:
"""
Healthcare Cost — Binary Classification Pipeline  v2
=====================================================
Target  : TotalCost > $30,000  →  HighCostLabel = 1
Primary : F1-Score (positive / minority class)
CV      : Stratified K-Fold (5 folds) — folds come from preprocess.py

Phases
  1   Setup & load preprocessed data
  2   Baseline models  (LR, NB, KNN, DT, Ridge)
  3   Ensemble models  (RF, ET, AdaBoost, GBM, Bagging)
  4   Gradient boosting  (XGBoost / LightGBM / CatBoost)
  5   Neural network  (MLP)
  6   SMOTE variants  (SMOTE / ADASYN / SMOTETomek)
  7   Optuna tuning  (top-3 by CV F1)
  8   Stacking + soft-voting ensemble
  9   Threshold calibration on held-out fold
 10   Final evaluation, plots, feature importance, model save
"""

import os, sys, warnings, time, joblib, json
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline        import Pipeline
from sklearn.preprocessing   import RobustScaler
from sklearn.impute          import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics         import (f1_score, precision_score, recall_score,
                                      roc_auc_score, average_precision_score,
                                      precision_recall_curve, roc_curve,
                                      confusion_matrix, classification_report,
                                      make_scorer)
from sklearn.linear_model    import LogisticRegression, RidgeClassifier
from sklearn.naive_bayes     import GaussianNB
from sklearn.tree            import DecisionTreeClassifier
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.ensemble        import (RandomForestClassifier, ExtraTreesClassifier,
                                      GradientBoostingClassifier, AdaBoostClassifier,
                                      BaggingClassifier, StackingClassifier, VotingClassifier)
from sklearn.neural_network  import MLPClassifier
from sklearn.calibration     import CalibratedClassifierCV

try:
    from imblearn.over_sampling import SMOTE, ADASYN
    from imblearn.combine       import SMOTETomek
    from imblearn.pipeline      import Pipeline as ImbPipeline
    IMBLEARN = True
except ImportError:
    IMBLEARN = False

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

try:
    import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING); OPTUNA_OK = True
except ImportError:
    OPTUNA_OK = False

warnings.filterwarnings("ignore")
C = dict(low="#4C9BE8", high="#E8654C", mid="#6DC78A", neutral="#9B9EBB", accent="#F5A623", bg="#F7F8FC")
matplotlib.rcParams.update({"figure.dpi":130,"axes.spines.top":False,"axes.spines.right":False})
OUT = "./model_outputs"; os.makedirs(OUT, exist_ok=True)

def save_fig(fig, name):
    fig.savefig(os.path.join(OUT, name), bbox_inches="tight", facecolor=C["bg"])
    plt.close(fig); print(f"    → {name}")


# ═══════════════════════════════════════════════════════════════════════════════
#  PHASE 1 — LOAD DATA FROM PREPROCESS
# ═══════════════════════════════════════════════════════════════════════════════
print("="*65)
print("  PHASE 1 — Data Loading")
print("="*65)

import importlib.util as _ilu
_s = _ilu.spec_from_file_location("preprocess", "./preprocess.py")
_pp = _ilu.module_from_spec(_s); _s.loader.exec_module(_pp)

BASE_DIR      = r"/home/noneo/Codes/ML/MachineLearning/Hand_On_ML/Data/Healthcare Cost Dataset"
TRAINING_PATH = os.path.join(BASE_DIR, "train", "train") + os.sep

main_df = pd.read_csv(TRAINING_PATH + "main_df_train.csv", low_memory=False)
cpt_df  = pd.read_csv(TRAINING_PATH + "cpt_df_train.csv",  low_memory=False)
drg_df  = pd.read_csv(TRAINING_PATH + "drg_df_train.csv",  low_memory=False)
icd_df  = pd.read_csv(TRAINING_PATH + "icd_df_train.csv",  low_memory=False)
dob_df  = pd.read_csv(TRAINING_PATH + "dob_df.csv",         low_memory=False)

X, y_reg, y_cls, skf_splits, merged = _pp.run_pipeline(
    main_df, cpt_df, drg_df, icd_df, dob_df
)

# Imbalance ratio — used by gradient boosters
pos_rate       = y_cls.mean()
neg_pos_ratio  = (1 - pos_rate) / pos_rate
FEAT_NAMES     = X.columns.tolist()
print(f"\n  Imbalance ratio (neg:pos) = {neg_pos_ratio:.1f}:1")
print(f"  Total features            = {len(FEAT_NAMES)}")


# ═══════════════════════════════════════════════════════════════════════════════
#  SHARED UTILITIES
# ═══════════════════════════════════════════════════════════════════════════════

# Pre-processing sub-pipeline (inside each model pipeline)
prep_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  RobustScaler()),
])

def make_pipe(clf):
    return Pipeline([("prep", prep_pipe), ("clf", clf)])

# Scoring dict for cross_validate
SCORING = {
    "f1"       : make_scorer(f1_score, zero_division=0),
    "precision": make_scorer(precision_score, zero_division=0),
    "recall"   : make_scorer(recall_score),
    "roc_auc"  : make_scorer(roc_auc_score, needs_proba=True),
    "pr_auc"   : make_scorer(average_precision_score, needs_proba=True),
}

# Pass pre-computed SKF splits to cross_validate via cv parameter
# sklearn accepts a list of (train_idx, val_idx) tuples
SKF_CV = skf_splits     # 5 folds, already stratified

RESULTS = {}

def cv_eval(name, pipe, X, y, verbose=True):
    """5-fold stratified CV using folds from preprocess.py."""
    t0  = time.time()
    try:
        cv  = cross_validate(pipe, X, y, cv=SKF_CV, scoring=SCORING,
                             return_train_score=False, n_jobs=-1)
        row = dict(
            F1        = cv["test_f1"].mean(),
            F1_std    = cv["test_f1"].std(),
            Precision = cv["test_precision"].mean(),
            Recall    = cv["test_recall"].mean(),
            ROC_AUC   = cv["test_roc_auc"].mean(),
            PR_AUC    = cv["test_pr_auc"].mean(),
            Time_s    = time.time() - t0,
            Status    = "OK",
        )
    except Exception as e:
        row = dict(F1=0,F1_std=0,Precision=0,Recall=0,ROC_AUC=0,PR_AUC=0,
                   Time_s=time.time()-t0, Status=str(e)[:60])
    RESULTS[name] = row
    if verbose:
        tag = "✓" if row["Status"]=="OK" else "✗"
        print(f"  {tag} {name:<40} F1={row['F1']:.4f}±{row['F1_std']:.4f}  "
              f"ROC={row['ROC_AUC']:.4f}  PR={row['PR_AUC']:.4f}  ({row['Time_s']:.1f}s)")
    return row


# ═══════════════════════════════════════════════════════════════════════════════
#  PHASE 2 — BASELINE MODELS
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("  PHASE 2 — Baseline Models")
print("="*65)

baselines = {
    "LR (L2, balanced)"   : LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
    "LR (L1, balanced)"   : LogisticRegression(penalty="l1", solver="saga", max_iter=2000,
                                                class_weight="balanced", random_state=42),
    "Naive Bayes"         : GaussianNB(),
    "KNN (k=5)"           : KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    "KNN (k=11)"          : KNeighborsClassifier(n_neighbors=11, n_jobs=-1),
    "Decision Tree (d=5)" : DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42),
    "Decision Tree (d=8)" : DecisionTreeClassifier(max_depth=8, class_weight="balanced", random_state=42),
    "Ridge Classifier"    : RidgeClassifier(class_weight="balanced"),
}
for name, clf in baselines.items():
    cv_eval(name, make_pipe(clf), X, y_cls)


# ═══════════════════════════════════════════════════════════════════════════════
#  PHASE 3 — ENSEMBLE MODELS
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("  PHASE 3 — Ensemble Models")
print("="*65)

ensembles = {
    "Random Forest (100)"        : RandomForestClassifier(n_estimators=100, class_weight="balanced",
                                                           n_jobs=-1, random_state=42),
    "Random Forest (300, d=12)"  : RandomForestClassifier(n_estimators=300, max_depth=12,
                                                           class_weight="balanced", n_jobs=-1, random_state=42),
    "Extra Trees (200)"          : ExtraTreesClassifier(n_estimators=200, class_weight="balanced",
                                                         n_jobs=-1, random_state=42),
    "AdaBoost (100)"             : AdaBoostClassifier(n_estimators=100, random_state=42),
    "GradBoost-sklearn (200)"    : GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                                                               max_depth=4, random_state=42),
    "Bagging + DT"               : BaggingClassifier(
                                       estimator=DecisionTreeClassifier(max_depth=8),
                                       n_estimators=50, n_jobs=-1, random_state=42),
}
for name, clf in ensembles.items():
    cv_eval(name, make_pipe(clf), X, y_cls)


# ═══════════════════════════════════════════════════════════════════════════════
#  PHASE 4 — GRADIENT BOOSTING
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("  PHASE 4 — Gradient Boosting")
print("="*65)

if XGB_OK:
    for name, kwargs in [
        ("XGBoost (default)",  dict(n_estimators=300, learning_rate=0.05, max_depth=6,
                                    subsample=0.8, colsample_bytree=0.8)),
        ("XGBoost (deep)",     dict(n_estimators=500, learning_rate=0.03, max_depth=8,
                                    subsample=0.7, colsample_bytree=0.7,
                                    min_child_weight=3, gamma=0.1, reg_alpha=0.1)),
        ("XGBoost (shallow)",  dict(n_estimators=400, learning_rate=0.05, max_depth=4,
                                    subsample=0.85, colsample_bytree=0.85)),
    ]:
        clf = XGBClassifier(**kwargs, scale_pos_weight=neg_pos_ratio,
                             use_label_encoder=False, eval_metric="logloss",
                             tree_method="hist", random_state=42, n_jobs=-1)
        cv_eval(name, make_pipe(clf), X, y_cls)

if LGB_OK:
    for name, kwargs in [
        ("LightGBM (default)", dict(n_estimators=300, learning_rate=0.05, num_leaves=63)),
        ("LightGBM (DART)",    dict(boosting_type="dart", n_estimators=200,
                                    learning_rate=0.05, num_leaves=31)),
        ("LightGBM (large)",   dict(n_estimators=500, learning_rate=0.03,
                                    num_leaves=127, min_child_samples=20)),
    ]:
        clf = LGBMClassifier(**kwargs, class_weight="balanced",
                              subsample=0.8, colsample_bytree=0.8,
                              random_state=42, n_jobs=-1, verbosity=-1)
        cv_eval(name, make_pipe(clf), X, y_cls)

if CAT_OK:
    for name, kwargs in [
        ("CatBoost (default)", dict(iterations=300, learning_rate=0.05, depth=6)),
        ("CatBoost (deep)",    dict(iterations=500, learning_rate=0.03, depth=8, l2_leaf_reg=5)),
    ]:
        clf = CatBoostClassifier(**kwargs, auto_class_weights="Balanced",
                                  random_state=42, verbose=0)
        cv_eval(name, make_pipe(clf), X, y_cls)


# ═══════════════════════════════════════════════════════════════════════════════
#  PHASE 5 — NEURAL NETWORK
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("  PHASE 5 — Neural Network (MLP)")
print("="*65)

for name, layers in [
    ("MLP (64-32)",      (64, 32)),
    ("MLP (128-64-32)",  (128, 64, 32)),
    ("MLP (256-128-64)", (256, 128, 64)),
]:
    clf = MLPClassifier(hidden_layer_sizes=layers, max_iter=400,
                        early_stopping=True, learning_rate_init=1e-3,
                        random_state=42)
    cv_eval(name, make_pipe(clf), X, y_cls)


# ═══════════════════════════════════════════════════════════════════════════════
#  PHASE 6 — SMOTE RESAMPLING
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("  PHASE 6 — SMOTE Resampling")
print("="*65)

if IMBLEARN:
    # Pick best available gradient booster as the base for SMOTE experiments
    if LGB_OK:
        _smote_base = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=63,
                                      class_weight="balanced", subsample=0.8,
                                      random_state=42, n_jobs=-1, verbosity=-1)
        _tag = "LightGBM"
    elif XGB_OK:
        _smote_base = XGBClassifier(n_estimators=300, scale_pos_weight=neg_pos_ratio,
                                     use_label_encoder=False, eval_metric="logloss",
                                     tree_method="hist", random_state=42, n_jobs=-1)
        _tag = "XGBoost"
    else:
        _smote_base = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                              n_jobs=-1, random_state=42)
        _tag = "RF"

    for sname, sampler in [
        ("SMOTE",      SMOTE(random_state=42)),
        ("ADASYN",     ADASYN(random_state=42)),
        ("SMOTETomek", SMOTETomek(random_state=42)),
    ]:
        pipe = ImbPipeline([
            ("prep",    prep_pipe),
            ("sampler", sampler),
            ("clf",     _smote_base),
        ])
        cv_eval(f"{_tag} + {sname}", pipe, X, y_cls)
else:
    print("  [SKIP] imbalanced-learn not installed")


# ═══════════════════════════════════════════════════════════════════════════════
#  PHASE 7 — OPTUNA HYPERPARAMETER TUNING
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("  PHASE 7 — Optuna Tuning  (top-3 by F1)")
print("="*65)

OPTUNA_BEST  = {}
TUNED_MODELS = {}     # name → fitted pipeline (on full data)

if OPTUNA_OK:
    # Prepare scaled array for fast Optuna callbacks
    _imp = SimpleImputer(strategy="median")
    _scl = RobustScaler()
    X_sc = _scl.fit_transform(_imp.fit_transform(X))
    y_np = y_cls.values

    def _fold_f1(clf):
        scores = []
        for tr, va in SKF_CV:
            clf.fit(X_sc[tr], y_np[tr])
            scores.append(f1_score(y_np[va], clf.predict(X_sc[va]), zero_division=0))
        return float(np.mean(scores))

    top3 = sorted(RESULTS, key=lambda k: RESULTS[k]["F1"], reverse=True)[:3]
    print(f"  Tuning: {top3}")

    # ── XGBoost ──────────────────────────────────────────────
    if XGB_OK and any("XGBoost" in n for n in top3):
        def _xgb_obj(trial):
            return _fold_f1(XGBClassifier(
                n_estimators     = trial.suggest_int("n_estimators",200,700),
                learning_rate    = trial.suggest_float("lr",0.005,0.15,log=True),
                max_depth        = trial.suggest_int("max_depth",3,10),
                subsample        = trial.suggest_float("subsample",0.5,1.0),
                colsample_bytree = trial.suggest_float("colsample_bytree",0.4,1.0),
                min_child_weight = trial.suggest_int("min_child_weight",1,15),
                gamma            = trial.suggest_float("gamma",0,0.8),
                reg_alpha        = trial.suggest_float("reg_alpha",0,2.0),
                reg_lambda       = trial.suggest_float("reg_lambda",0.5,5.0),
                scale_pos_weight = neg_pos_ratio,
                use_label_encoder=False, eval_metric="logloss",
                tree_method="hist", random_state=42, n_jobs=-1,
            ))
        study = optuna.create_study(direction="maximize",
                                     sampler=optuna.samplers.TPESampler(seed=42))
        study.optimize(_xgb_obj, n_trials=50, show_progress_bar=False)
        OPTUNA_BEST["XGBoost"] = study.best_params
        best_xgb = XGBClassifier(**{**study.best_params,
                                     "scale_pos_weight":neg_pos_ratio,
                                     "use_label_encoder":False,
                                     "eval_metric":"logloss",
                                     "tree_method":"hist",
                                     "random_state":42,"n_jobs":-1})
        cv_eval("XGBoost (Optuna)", make_pipe(best_xgb), X, y_cls)
        TUNED_MODELS["XGBoost (Optuna)"] = make_pipe(best_xgb)
        print(f"    Best XGB params: {study.best_params}")

    # ── LightGBM ─────────────────────────────────────────────
    if LGB_OK and any("LightGBM" in n for n in top3):
        def _lgb_obj(trial):
            return _fold_f1(LGBMClassifier(
                n_estimators     = trial.suggest_int("n_estimators",200,700),
                learning_rate    = trial.suggest_float("lr",0.005,0.15,log=True),
                num_leaves       = trial.suggest_int("num_leaves",20,200),
                max_depth        = trial.suggest_int("max_depth",4,15),
                subsample        = trial.suggest_float("subsample",0.5,1.0),
                colsample_bytree = trial.suggest_float("colsample_bytree",0.4,1.0),
                min_child_samples= trial.suggest_int("min_child_samples",5,100),
                reg_alpha        = trial.suggest_float("reg_alpha",0,2.0),
                reg_lambda       = trial.suggest_float("reg_lambda",0,2.0),
                class_weight     = "balanced",
                random_state=42, n_jobs=-1, verbosity=-1,
            ))
        study = optuna.create_study(direction="maximize",
                                     sampler=optuna.samplers.TPESampler(seed=42))
        study.optimize(_lgb_obj, n_trials=50, show_progress_bar=False)
        OPTUNA_BEST["LightGBM"] = study.best_params
        best_lgb = LGBMClassifier(**{**study.best_params,
                                      "class_weight":"balanced",
                                      "random_state":42,"n_jobs":-1,"verbosity":-1})
        cv_eval("LightGBM (Optuna)", make_pipe(best_lgb), X, y_cls)
        TUNED_MODELS["LightGBM (Optuna)"] = make_pipe(best_lgb)
        print(f"    Best LGB params: {study.best_params}")

    # ── CatBoost ─────────────────────────────────────────────
    if CAT_OK and any("CatBoost" in n for n in top3):
        def _cat_obj(trial):
            return _fold_f1(CatBoostClassifier(
                iterations          = trial.suggest_int("iterations",200,600),
                learning_rate       = trial.suggest_float("lr",0.005,0.15,log=True),
                depth               = trial.suggest_int("depth",4,10),
                l2_leaf_reg         = trial.suggest_float("l2_leaf_reg",1,15),
                bagging_temperature = trial.suggest_float("bagging_temperature",0,1),
                border_count        = trial.suggest_int("border_count",32,255),
                auto_class_weights  = "Balanced",
                random_state=42, verbose=0,
            ))
        study = optuna.create_study(direction="maximize",
                                     sampler=optuna.samplers.TPESampler(seed=42))
        study.optimize(_cat_obj, n_trials=35, show_progress_bar=False)
        OPTUNA_BEST["CatBoost"] = study.best_params
        best_cat = CatBoostClassifier(**{**study.best_params,
                                          "auto_class_weights":"Balanced",
                                          "random_state":42,"verbose":0})
        cv_eval("CatBoost (Optuna)", make_pipe(best_cat), X, y_cls)
        TUNED_MODELS["CatBoost (Optuna)"] = make_pipe(best_cat)
        print(f"    Best CAT params: {study.best_params}")

    with open(os.path.join(OUT,"optuna_best_params.json"),"w") as f:
        json.dump(OPTUNA_BEST, f, indent=2)
else:
    print("  [SKIP] Optuna not installed")


# ═══════════════════════════════════════════════════════════════════════════════
#  PHASE 8 — STACKING + SOFT VOTING
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("  PHASE 8 — Stacking & Soft Voting Ensemble")
print("="*65)

base_est = []
if LGB_OK:
    base_est.append(("lgb", LGBMClassifier(n_estimators=300, class_weight="balanced",
                                             subsample=0.8, random_state=42, n_jobs=-1, verbosity=-1)))
if XGB_OK:
    base_est.append(("xgb", XGBClassifier(n_estimators=300, scale_pos_weight=neg_pos_ratio,
                                            use_label_encoder=False, eval_metric="logloss",
                                            tree_method="hist", random_state=42, n_jobs=-1)))
if CAT_OK:
    base_est.append(("cat", CatBoostClassifier(iterations=300, auto_class_weights="Balanced",
                                                 random_state=42, verbose=0)))
base_est.append(("rf",  RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                                 n_jobs=-1, random_state=42)))
base_est.append(("et",  ExtraTreesClassifier(n_estimators=200, class_weight="balanced",
                                               n_jobs=-1, random_state=42)))

if len(base_est) >= 2:
    stacker = StackingClassifier(
        estimators      = base_est,
        final_estimator = LogisticRegression(max_iter=1000, class_weight="balanced"),
        cv              = SKF_CV,
        passthrough     = False,
        n_jobs          = -1,
    )
    cv_eval("Stacking Ensemble", make_pipe(stacker), X, y_cls)

    voter = VotingClassifier(estimators=base_est, voting="soft", n_jobs=-1)
    cv_eval("Soft Voting Ensemble", make_pipe(voter), X, y_cls)


# ═══════════════════════════════════════════════════════════════════════════════
#  PHASE 9 — THRESHOLD CALIBRATION  (on held-out fold 5)
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("  PHASE 9 — Threshold Calibration (held-out fold)")
print("="*65)

# Identify overall best model
best_name = max(RESULTS, key=lambda k: RESULTS[k]["F1"])
print(f"  Best model : {best_name}  (CV F1 = {RESULTS[best_name]['F1']:.4f})")

# Rebuild the best classifier
def _make_best_clf():
    if "XGBoost (Optuna)" in best_name and XGB_OK and OPTUNA_OK:
        return XGBClassifier(**{**OPTUNA_BEST.get("XGBoost",{}),
                                 "scale_pos_weight":neg_pos_ratio,
                                 "use_label_encoder":False,"eval_metric":"logloss",
                                 "tree_method":"hist","random_state":42,"n_jobs":-1})
    if "LightGBM (Optuna)" in best_name and LGB_OK and OPTUNA_OK:
        return LGBMClassifier(**{**OPTUNA_BEST.get("LightGBM",{}),
                                  "class_weight":"balanced",
                                  "random_state":42,"n_jobs":-1,"verbosity":-1})
    if "CatBoost (Optuna)" in best_name and CAT_OK and OPTUNA_OK:
        return CatBoostClassifier(**{**OPTUNA_BEST.get("CatBoost",{}),
                                      "auto_class_weights":"Balanced","random_state":42,"verbose":0})
    if "LightGBM" in best_name and LGB_OK:
        return LGBMClassifier(n_estimators=300,class_weight="balanced",
                               random_state=42,n_jobs=-1,verbosity=-1)
    if "XGBoost"  in best_name and XGB_OK:
        return XGBClassifier(n_estimators=300,scale_pos_weight=neg_pos_ratio,
                              use_label_encoder=False,eval_metric="logloss",
                              tree_method="hist",random_state=42,n_jobs=-1)
    if "CatBoost" in best_name and CAT_OK:
        return CatBoostClassifier(iterations=300,auto_class_weights="Balanced",
                                   random_state=42,verbose=0)
    return RandomForestClassifier(n_estimators=300,class_weight="balanced",
                                   n_jobs=-1,random_state=42)

best_pipe = make_pipe(_make_best_clf())

# Use fold 4 (last fold) as the calibration holdout
tr_idx, va_idx = SKF_CV[-1]
X_tr_f, y_tr_f = X.iloc[tr_idx], y_cls.iloc[tr_idx]
X_va_f, y_va_f = X.iloc[va_idx], y_cls.iloc[va_idx]

best_pipe.fit(X_tr_f, y_tr_f)
y_prob_va = best_pipe.predict_proba(X_va_f)[:, 1]

# Threshold sweep
thresholds  = np.arange(0.05, 0.95, 0.005)
thr_results = []
for thr in thresholds:
    pred = (y_prob_va >= thr).astype(int)
    thr_results.append(dict(
        threshold = thr,
        F1        = f1_score(y_va_f, pred, zero_division=0),
        Precision = precision_score(y_va_f, pred, zero_division=0),
        Recall    = recall_score(y_va_f, pred),
    ))
thr_df   = pd.DataFrame(thr_results)
best_row = thr_df.loc[thr_df["F1"].idxmax()]
OPT_THR  = round(best_row["threshold"], 3)
print(f"  Optimal threshold = {OPT_THR:.3f}  "
      f"F1={best_row['F1']:.4f}  P={best_row['Precision']:.4f}  R={best_row['Recall']:.4f}")

y_pred_thr = (y_prob_va >= OPT_THR).astype(int)

# Record calibrated result
RESULTS[f"{best_name} + thr={OPT_THR}"] = dict(
    F1        = f1_score(y_va_f, y_pred_thr, zero_division=0),
    F1_std    = 0.0,
    Precision = precision_score(y_va_f, y_pred_thr, zero_division=0),
    Recall    = recall_score(y_va_f, y_pred_thr),
    ROC_AUC   = roc_auc_score(y_va_f, y_prob_va),
    PR_AUC    = average_precision_score(y_va_f, y_prob_va),
    Time_s    = 0.0, Status="OK",
)


# ═══════════════════════════════════════════════════════════════════════════════
#  PHASE 10 — RESULTS, PLOTS, FEATURE IMPORTANCE, SAVE
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("  PHASE 10 — Final Evaluation")
print("="*65)

res_df = pd.DataFrame(RESULTS).T.sort_values("F1", ascending=False)
res_df.to_csv(os.path.join(OUT,"all_model_results.csv"))

print("\n  ── All Models Ranked by F1 ──")
print(res_df[["F1","F1_std","Precision","Recall","ROC_AUC","PR_AUC"]].to_string(float_format="{:.4f}".format))

# ── Plot 1: F1 comparison bar ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, max(7, len(res_df)*0.27)),
                          facecolor=C["bg"])
fig.suptitle("Model Comparison — All Models", fontsize=14, fontweight="bold")
plot_df = res_df.sort_values("F1")
colors  = [C["high"] if i >= len(plot_df)-3 else
           C["accent"] if i >= len(plot_df)-8 else C["low"]
           for i in range(len(plot_df))]
axes[0].barh(plot_df.index, plot_df["F1"],
             xerr=plot_df["F1_std"].clip(0),
             color=colors, error_kw=dict(ecolor=C["neutral"], lw=1))
axes[0].set_xlabel("F1 Score"); axes[0].set_title("F1 Score (5-fold SKF)")
axes[0].axvline(0.5, color=C["neutral"], lw=0.8, ls="--")
axes[1].barh(plot_df.index, plot_df["PR_AUC"], color=colors, alpha=0.85)
axes[1].set_xlabel("PR-AUC"); axes[1].set_title("PR-AUC (Average Precision)")
plt.tight_layout(); save_fig(fig, "10a_model_comparison.png")

# ── Plot 2: ROC + PR + threshold sweep ───────────────────────────────────────
fpr, tpr, _ = roc_curve(y_va_f, y_prob_va)
prec, rec, _ = precision_recall_curve(y_va_f, y_prob_va)
roc_auc_v    = roc_auc_score(y_va_f, y_prob_va)
pr_auc_v     = average_precision_score(y_va_f, y_prob_va)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor=C["bg"])
fig.suptitle(f"Best Model: {best_name[:55]}", fontsize=11, fontweight="bold")
axes[0].plot(fpr, tpr, color=C["high"], lw=2, label=f"AUC={roc_auc_v:.4f}")
axes[0].plot([0,1],[0,1], color=C["neutral"], lw=1, ls="--")
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR"); axes[0].set_title("ROC Curve")
axes[0].legend()
axes[1].plot(rec, prec, color=C["low"], lw=2, label=f"AP={pr_auc_v:.4f}")
axes[1].axhline(y_va_f.mean(), color=C["neutral"], lw=1, ls="--", label="Baseline")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve"); axes[1].legend()
axes[2].plot(thr_df["threshold"], thr_df["F1"],        color=C["high"],   label="F1")
axes[2].plot(thr_df["threshold"], thr_df["Precision"], color=C["low"],    label="Precision")
axes[2].plot(thr_df["threshold"], thr_df["Recall"],    color=C["accent"], label="Recall")
axes[2].axvline(OPT_THR, color="black", lw=1.5, ls="--", label=f"thr={OPT_THR}")
axes[2].set_xlabel("Threshold"); axes[2].set_title("Threshold Sweep"); axes[2].legend(fontsize=8)
plt.tight_layout(); save_fig(fig, "10b_roc_pr_threshold.png")

# ── Plot 3: Confusion matrix ──────────────────────────────────────────────────
cm      = confusion_matrix(y_va_f, y_pred_thr)
cm_norm = cm / cm.sum(axis=1, keepdims=True)
fig, axes = plt.subplots(1, 2, figsize=(10,4), facecolor=C["bg"])
fig.suptitle(f"Confusion Matrix — thr={OPT_THR}", fontsize=11, fontweight="bold")
for ax, data, fmt, title in [
    (axes[0], cm,      "d",   "Raw counts"),
    (axes[1], cm_norm, ".2%", "Row-normalised"),
]:
    sns.heatmap(data, annot=True, fmt=fmt, cmap="RdYlGn", ax=ax,
                xticklabels=["Low-Cost","High-Cost"],
                yticklabels=["Low-Cost","High-Cost"], linewidths=0.5)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(title)
plt.tight_layout(); save_fig(fig, "10c_confusion_matrix.png")

# ── Plot 4: Feature importance ────────────────────────────────────────────────
_clf = best_pipe.named_steps["clf"]
importances = None
if hasattr(_clf, "feature_importances_"):
    importances = _clf.feature_importances_
elif hasattr(_clf, "coef_"):
    importances = np.abs(_clf.coef_[0])

if importances is not None and len(importances) == len(FEAT_NAMES):
    fi_df = (pd.DataFrame({"feature":FEAT_NAMES,"importance":importances})
               .sort_values("importance", ascending=False).head(30))
    fi_df.to_csv(os.path.join(OUT,"feature_importance.csv"), index=False)
    fig, ax = plt.subplots(figsize=(10, 9), facecolor=C["bg"])
    grad = [C["high"] if i<5 else C["accent"] if i<15 else C["low"] for i in range(len(fi_df))]
    ax.barh(fi_df["feature"][::-1], fi_df["importance"][::-1], color=grad[::-1])
    ax.set_title(f"Top-30 Feature Importances — {best_name[:40]}")
    ax.set_xlabel("Importance")
    plt.tight_layout(); save_fig(fig, "10d_feature_importance.png")

# ── Classification report ─────────────────────────────────────────────────────
print("\n  ── Classification Report (held-out fold, tuned threshold) ──")
print(classification_report(y_va_f, y_pred_thr, target_names=["Low-Cost","High-Cost"]))

# ── Save final model ──────────────────────────────────────────────────────────
# Retrain best pipeline on ALL data before saving
best_pipe_full = make_pipe(_make_best_clf())
best_pipe_full.fit(X, y_cls)
save_payload = dict(pipeline=best_pipe_full, threshold=OPT_THR, feature_names=FEAT_NAMES)
joblib.dump(save_payload, os.path.join(OUT,"best_model.pkl"))

print(f"\n{'='*65}")
print(f"  PIPELINE COMPLETE")
print(f"  Best model  : {best_name}")
print(f"  CV F1       : {RESULTS[best_name]['F1']:.4f}")
print(f"  Hold-out F1 : {f1_score(y_va_f, y_pred_thr, zero_division=0):.4f}  (fold-5, thr={OPT_THR})")
print(f"  ROC-AUC     : {roc_auc_v:.4f}")
print(f"  PR-AUC      : {pr_auc_v:.4f}")
print(f"  Saved →     {OUT}/best_model.pkl")
print(f"{'='*65}")

  PHASE 1 — Data Loading

  Processing full training set
    main_df → 45,394 rows × 293 cols
    dob_df  → 64,443 rows × 4 cols
    cpt_df  → 162,526 rows after aggregation
    drg_df  → 230,814 rows after aggregation
    icd_df  → 162,525 rows after aggregation
    merged  → 45,394 rows × 309 cols

  ✓  X          : (45394, 306)
  ✓  y_reg      : min=524  median=3887  max=922,005
  ✓  y_cls      : 4,058 positive / 45,394 total  (8.9%)
  ✓  SKF folds  : 5  (each val fold ≈ 9,078 rows)

  Fold  |  Train+  Train-  |  Val+   Val-   |  Val pos%
  ─────────────────────────────────────────────────────
  1     |   3,247  33,068  |    811   8,268  |  8.9%
  2     |   3,246  33,069  |    812   8,267  |  8.9%
  3     |   3,246  33,069  |    812   8,267  |  8.9%
  4     |   3,246  33,069  |    812   8,267  |  8.9%
  5     |   3,247  33,069  |    811   8,267  |  8.9%

  Imbalance ratio (neg:pos) = 10.2:1
  Total features            = 306

  PHASE 2 — Baseline Models


/home/noneo/miniconda3/envs/aienv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/noneo/miniconda3/envs/aienv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_

  ✓ LR (L2, balanced)                        F1=0.7005±0.0062  ROC=0.9858  PR=0.9236  (147.6s)


/home/noneo/miniconda3/envs/aienv/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/noneo/miniconda3/envs/aienv/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/noneo/miniconda3/envs/aienv/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/noneo/miniconda3/envs/aienv/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/noneo/miniconda3/envs/aienv/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


  ✓ LR (L1, balanced)                        F1=0.2905±0.0104  ROC=0.9392  PR=0.8482  (379.7s)
  ✓ Naive Bayes                              F1=0.8111±0.0064  ROC=0.9822  PR=0.7456  (6.6s)
